# BERT Sentiment Classification with SST-2

This notebook evaluates a pretrained BERT sentiment-classification model on a sampled SST-2 validation set, visualizes the confusion matrix, reviews high-confidence mistakes, and performs custom sentence inference.

## 1. Install Dependencies

In [ ]:
%pip install -q -U transformers datasets accelerate

## 2. Imports, Environment Setup, and Model Loading

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

### Set random seed for reproducibility

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Initialize model and tokenizer

In [ ]:
MODEL_NAME = "textattack/bert-base-uncased-SST-2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()
print("Device:", device)

## 3. Load and Sample Validation Dataset

In [ ]:
dataset = load_dataset("nyu-mll/glue", "sst2", split="validation")
sample = dataset.shuffle(seed=SEED).select(range(min(500, len(dataset))))
display(pd.DataFrame(sample[:5]))

## 4. Prediction Batch Helper Function

In [ ]:
@torch.no_grad()
def predict_texts(texts, batch_size=32):
    predictions, positive_probabilities = [], []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)
        probabilities = model(**encoded).logits.softmax(dim=-1)
        predictions.extend(probabilities.argmax(dim=-1).cpu().tolist())
        positive_probabilities.extend(probabilities[:, 1].cpu().tolist())
    return np.array(predictions), np.array(positive_probabilities)

## 5. Evaluate Model Performance

In [ ]:
texts = sample["sentence"]
y_true = np.array(sample["label"])
y_pred, positive_probability = predict_texts(texts)

accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="binary", zero_division=0
)

print(
    pd.Series({
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }).round(3)
)

## 6. Confusion Matrix Visualization

In [ ]:
cm = confusion_matrix(y_true, y_pred)

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Purples",
    xticklabels=["negative", "positive"],
    yticklabels=["negative", "positive"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("BERT confusion matrix")
plt.show()

## 7. Analyze Top Mistakes and Misclassifications

In [ ]:
review = pd.DataFrame({
    "text": texts,
    "actual": y_true,
    "predicted": y_pred,
    "positive_probability": positive_probability,
})

mistakes = review[review.actual != review.predicted].copy()

mistakes["confidence"] = np.where(
    mistakes.predicted == 1,
    mistakes.positive_probability,
    1 - mistakes.positive_probability,
)

display(mistakes.sort_values("confidence", ascending=False).head(10))

## 8. Custom Sentence Inference

In [ ]:
custom_text = ["The story was slow, but the ending was wonderful."]

label, probability = predict_texts(custom_text)

print(
    custom_text[0],
    "->",
    ["negative", "positive"][label[0]],
    f"(positive probability={probability[0]:.3f})"
)